# Named-entity recognition and clinical coding

Exercise notebook for the lesson **[Named-entity recognition and clinical coding](https://ramkumarmd.com/medical-ai/clinical-nlp/clinical-ner-and-coding)** in the *Clinical NLP* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Named-entity recognition (NER) finds mentions of things such as conditions, medicines and procedures in text; normalisation maps each mention to a standard code, such as ICD-10 or SNOMED CT. Dictionary methods are transparent and fast but miss synonyms and spelling variants; machine-learning and transformer models find more but need labelled data and validation. Any extractor must handle negation and be scored against human annotations with precision and recall. Studies have found that large language models are unreliable at assigning medical codes directly.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### A dictionary extractor with negation

Dictionary-based extraction to ICD-10 categories, scored with and without negation handling.

In [ ]:
import re

# Illustrative WHO ICD-10 categories; always confirm codes in the official classification
LEXICON = {
    "E11": ["type 2 diabetes", "t2dm", "type ii diabetes"],
    "I10": ["hypertension", "htn", "high blood pressure"],
    "N18": ["chronic kidney disease", "ckd"],
    "L97": ["foot ulcer", "ulcer of the foot", "ulcers"],
    "E78": ["hyperlipidaemia", "dyslipidaemia", "high cholesterol"],
}
NEGATION = r"\b(no|denies|without|negative for|absent)\b"

def extract_codes(note, use_negation=True):
    codes = set()
    for sentence in re.split(r"(?<=[.!?])\s+", note.lower()):
        for code, terms in LEXICON.items():
            for term in terms:
                m = re.search(rf"\b{re.escape(term)}\b", sentence)
                if m and not (use_negation and re.search(NEGATION, sentence[:m.start()])):
                    codes.add(code)
    return codes

NOTES = [   # synthetic notes with the codes a (fictional) clinical coder assigned
    ("58M with T2DM and HTN. No foot ulcers. Neuropathy on monofilament testing.", {"E11", "I10"}),
    ("Type 2 diabetes, CKD stage 3. Denies chest pain. High cholesterol on statin.", {"E11", "N18", "E78"}),
    ("Known DM2 for 6 years. Left heel ulcer, 2 cm, no signs of infection.", {"E11", "L97"}),
    ("Hypertension well controlled. Negative for diabetes on OGTT.", {"I10"}),
]

def score(use_negation):
    tp = fp = fn = 0
    for text, gold in NOTES:
        pred = extract_codes(text, use_negation)
        tp, fp, fn = tp + len(pred & gold), fp + len(pred - gold), fn + len(gold - pred)
    return tp / (tp + fp), tp / (tp + fn)

for label, neg in [("Without negation handling", False), ("With negation handling", True)]:
    precision, recall = score(neg)
    print(f"{label:<27} precision {precision:.2f}  recall {recall:.2f}")

# Without negation handling   precision 0.86  recall 0.75
# With negation handling      precision 1.00  recall 0.75

### Where dictionaries fail

Continuing from the previous block: list the notes where extraction disagrees with the coder.

In [ ]:
for text, gold in NOTES:
    pred = extract_codes(text)
    if pred != gold:
        print(f"{text}\n  missed {sorted(gold - pred)}  extra {sorted(pred - gold)}")

# Known DM2 for 6 years. Left heel ulcer, 2 cm, no signs of infection.
#   missed ['E11', 'L97']  extra []

## Exercise data

Synthetic data written for this exercise (no real patients). Run this cell before starting the steps.

In [ ]:
# Thirty SYNTHETIC notes with the codes a (fictional) clinical coder assigned, using the five illustrative
# ICD-10 categories of the lesson's lexicon (E11, I10, N18, L97, E78). No real patients.
CODED_NOTES = [
    ("58M with T2DM and HTN. No foot ulcers.", {"E11", "I10"}),
    ("Type 2 diabetes, CKD stage 3. High cholesterol on statin.", {"E11", "N18", "E78"}),
    ("Known DM2 for 6 years. Left heel ulcer, 2 cm.", {"E11", "L97"}),
    ("Hypertension well controlled. Negative for diabetes on OGTT.", {"I10"}),
    ("Type II diabetes mellitus. eGFR 48, chronic kidney disease.", {"E11", "N18"}),
    ("Raised BP readings, diagnosed hypertension today.", {"I10"}),
    ("No history of diabetes. Dyslipidaemia, started atorvastatin.", {"E78"}),
    ("Foot ulcer, right hallux, healing. T2DM.", {"E11", "L97"}),
    ("T2DM on metformin. Denies high blood pressure. Lipids normal.", {"E11"}),
    ("Mother has type 2 diabetes. Patient's HbA1c 5.4%, normal.", set()),
    ("Possible CKD, repeat eGFR in 3 months.", set()),
    ("Type 2 DM with nephropathy, CKD 4.", {"E11", "N18"}),
    ("Essential hypertension. Hyperlipidaemia.", {"I10", "E78"}),
    ("Ulceration of the left foot over the 5th metatarsal head. T2DM.", {"E11", "L97"}),
    ("Type 2 diabetes. HTN. CKD 3a. High cholesterol.", {"E11", "I10", "N18", "E78"}),
    ("Pre-diabetes on HbA1c. No hypertension.", set()),
    ("Non-insulin-dependent diabetes since 2015.", {"E11"}),
    ("Raised cholesterol, family history of heart disease.", {"E78"}),
    ("Kidney function reduced, stage 3 chronic kidney disease. No diabetes.", {"N18"}),
    ("Neuropathic ulcer on the sole of the right foot. Type 2 diabetes.", {"E11", "L97"}),
    ("Father had hypertension. Patient's BP 118/76.", set()),
    ("T2DM, well controlled. Foot check: no ulcers, pulses present.", {"E11"}),
    ("HTN on amlodipine. Query CKD, awaiting urine ACR.", {"I10"}),
    ("Type 2 diabetes with foot ulcer and high blood pressure.", {"E11", "L97", "I10"}),
    ("Hypercholesterolaemia. Statin declined.", {"E78"}),
    ("Diabetes type 2, insulin-treated. Chronic renal failure stage 4.", {"E11", "N18"}),
    ("Rule out hypertension: home BP monitoring arranged.", set()),
    ("DM2. HTN. Dyslipidemia.", {"E11", "I10", "E78"}),
    ("Ulcer on the heel, infected. Type 2 diabetes.", {"E11", "L97"}),
    ("Patient without diabetes. CKD stage 2 from polycystic kidneys.", {"N18"}),
]
print(f"{len(CODED_NOTES)} coded synthetic notes")

## Exercise

**Goal:** Build, score and improve a clinical entity extractor on synthetic notes.

**Data:** Thirty synthetic notes with coder-style labels, written for the notebook (Created for this course; no real patient data)

### Step 1

Reproduce the extractor and scores, then run it on the thirty notes and report precision and recall per code.

In [ ]:
# Your code here


### Step 2

Add synonyms and abbreviations from your error list and re-score; did precision fall?

In [ ]:
# Your code here


### Step 3

Add uncertainty and family-history handling from the previous lesson and measure the effect.

In [ ]:
# Your code here


### Step 4

Optional: run a scispaCy or medspaCy model on the same notes and compare.

In [ ]:
# Your code here


### Step 5

Optional, with an approved model: ask an LLM to extract conditions (not codes), map them to codes with your lexicon, and compare with direct LLM coding.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/clinical-nlp/clinical-ner-and-coding](https://ramkumarmd.com/medical-ai/clinical-nlp/clinical-ner-and-coding). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.